In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
df = pd.read_csv('/kaggle/working/SIEM_data.csv')

In [3]:
df.columns

Index(['Unnamed: 0', 'AlertTitle', 'Sha256', 'IpAddress', 'Url', 'AccountUpn',
       'AccountName', 'DeviceName', 'RegistryKey', 'RegistryValueName',
       'RegistryValueData', 'ApplicationName', 'FileName', 'FolderPath',
       'ResourceIdName', 'OSFamily', 'OSVersion', 'SuspicionLevel',
       'SuspicionLevel_label'],
      dtype='object')

In [4]:
TARGET = 'SuspicionLevel_label'
features_to_drop = ['Unnamed: 0', 'SuspicionLevel', TARGET]
X = df.drop(columns=features_to_drop)
y = df[TARGET]

In [5]:
X.columns

Index(['AlertTitle', 'Sha256', 'IpAddress', 'Url', 'AccountUpn', 'AccountName',
       'DeviceName', 'RegistryKey', 'RegistryValueName', 'RegistryValueData',
       'ApplicationName', 'FileName', 'FolderPath', 'ResourceIdName',
       'OSFamily', 'OSVersion'],
      dtype='object')

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

In [7]:
print(f"Training set size: {len(X_train)} samples")
print(f"Testing set size: {len(X_test)} samples")

Training set size: 293567 samples
Testing set size: 125815 samples


In [8]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [9]:
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

scaler_filename = 'standard_scaler.pkl'
joblib.dump(scaler, scaler_filename)
print(f"StandardScaler saved successfully to {scaler_filename}")

StandardScaler saved successfully to standard_scaler.pkl


In [10]:
knn_clf = KNeighborsClassifier(n_neighbors=5, n_jobs=-1)
dt_clf = DecisionTreeClassifier(random_state=42)
rf_clf = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)

In [11]:
voting_clf = VotingClassifier(estimators=[('knn', knn_clf),('dt', dt_clf),('rf', rf_clf)],voting='soft',n_jobs=-1)

In [12]:
voting_clf

VotingClassifier(estimators=[('knn', KNeighborsClassifier(n_jobs=-1)),
                             ('dt', DecisionTreeClassifier(random_state=42)),
                             ('rf',
                              RandomForestClassifier(n_estimators=50, n_jobs=-1,
                                                     random_state=42))],
                 n_jobs=-1, voting='soft')

In [13]:
voting_clf.fit(X_train_scaled, y_train)

VotingClassifier(estimators=[('knn', KNeighborsClassifier(n_jobs=-1)),
                             ('dt', DecisionTreeClassifier(random_state=42)),
                             ('rf',
                              RandomForestClassifier(n_estimators=50, n_jobs=-1,
                                                     random_state=42))],
                 n_jobs=-1, voting='soft')

In [14]:
y_pred = voting_clf.predict(X_test_scaled)

In [15]:
accuracy = accuracy_score(y_test, y_pred)
print(f"\nVotingClassifier Accuracy Score: {accuracy:.4f}")


VotingClassifier Accuracy Score: 0.9982


In [16]:
print("\nVotingClassifier Classification Report:")
print(classification_report(y_test, y_pred, zero_division=0))


VotingClassifier Classification Report:
              precision    recall  f1-score   support

           0       0.08      0.05      0.07       147
           1       1.00      1.00      1.00    125668

    accuracy                           1.00    125815
   macro avg       0.54      0.53      0.53    125815
weighted avg       1.00      1.00      1.00    125815



In [17]:
cm = confusion_matrix(y_test, y_pred)
print("\nVotingClassifier Confusion Matrix:")
print(cm)


VotingClassifier Confusion Matrix:
[[     8    139]
 [    89 125579]]


In [18]:
model_filename = 'voting_classifier_model.pkl'
joblib.dump(voting_clf, model_filename)
print(f"Trained VotingClassifier saved successfully to {model_filename}")

Trained VotingClassifier saved successfully to voting_classifier_model.pkl


In [19]:
!pip freeze > requirements.txt

In [20]:
!python --version

Python 3.11.13
